# 🧠 Intelligent App Testing System
### Data Science in SDLC — Bug Risk Analysis & Prediction
---
**Goal:** Use data science to predict which modules/bugs are high-risk so testers can prioritize smarter.


## 📦 Step 1: Import Libraries
We import all the tools we need for data handling, visualization, and machine learning.

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Text Analysis (NLP)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('✅ All libraries imported successfully!')

## 📂 Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('../data/bug_dataset.csv')

print(f'Shape: {df.shape}')   # rows x columns
print(f'Columns: {list(df.columns)}')
df.head(10)

## 🧹 Step 3: Data Cleaning & Preprocessing
We handle missing values, convert data types, and standardize fields.

In [ ]:
# ── 3a. Check missing values ──────────────────────────────────────
print('Missing values before cleaning:')
print(df.isnull().sum())

In [ ]:
# ── 3b. Fill missing Severity with 'Medium' (most common) ─────────
df['Severity'].fillna('Medium', inplace=True)

# ── 3c. Fill missing Time_to_Fix with median ─────────────────────
df['Time_to_Fix_Days'].fillna(df['Time_to_Fix_Days'].median(), inplace=True)

# ── 3d. Convert Release_Date to datetime ─────────────────────────
df['Release_Date'] = pd.to_datetime(df['Release_Date'])

# ── 3e. Standardize text fields ──────────────────────────────────
df['Severity'] = df['Severity'].str.strip().str.capitalize()
df['Status']   = df['Status'].str.strip().str.capitalize()
df['Module']   = df['Module'].str.strip()

# ── 3f. Extract Year & Month for trend analysis ───────────────────
df['Release_Year']  = df['Release_Date'].dt.year
df['Release_Month'] = df['Release_Date'].dt.month

print('✅ Cleaning complete. Missing values after:')
print(df.isnull().sum())
df.head()

In [ ]:
# Save clean dataset
df.to_csv('../data/clean_bug_dataset.csv', index=False)
print('✅ Clean dataset saved!')

## 📊 Step 4: Exploratory Data Analysis (EDA)
We explore the data visually to find patterns and trends.

In [ ]:
# ── 4a. Bug count per Module ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

module_counts = df['Module'].value_counts()
module_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('🔴 Bug Count per Module', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Module')
axes[0].set_ylabel('Number of Bugs')
axes[0].tick_params(axis='x', rotation=45)

# ── 4b. Severity Distribution ─────────────────────────────────────
severity_colors = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#2ecc71'}
df['Severity'].value_counts().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=[severity_colors.get(s, 'gray') for s in df['Severity'].value_counts().index],
    startangle=90
)
axes[1].set_title('🟠 Severity Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/eda_module_severity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

In [ ]:
# ── 4c. Status distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

status_colors = ['#3498db', '#2ecc71', '#e74c3c']
df['Status'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=status_colors, edgecolor='black'
)
axes[0].set_title('📋 Bug Status Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# ── 4d. Bugs per App Version ──────────────────────────────────────
version_order = sorted(df['App_Version'].unique())
version_counts = df['App_Version'].value_counts().reindex(version_order)
version_counts.plot(kind='line', ax=axes[1], marker='o', color='purple', linewidth=2)
axes[1].set_title('📈 Bug Count Across Versions', fontsize=14, fontweight='bold')
axes[1].set_xlabel('App Version')
axes[1].set_ylabel('Number of Bugs')

plt.tight_layout()
plt.savefig('../data/eda_status_version.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4e. Heatmap — Module vs Severity ─────────────────────────────
heatmap_data = pd.crosstab(df['Module'], df['Severity'])

# Reorder severity columns
for col in ['Low', 'Medium', 'High']:
    if col not in heatmap_data.columns:
        heatmap_data[col] = 0
heatmap_data = heatmap_data[['Low', 'Medium', 'High']]

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, linecolor='gray')
plt.title('🔥 Heatmap: Module vs Severity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4f. Time to Fix distribution ─────────────────────────────────
plt.figure(figsize=(10, 4))
sns.histplot(df['Time_to_Fix_Days'], bins=30, kde=True, color='teal')
plt.title('⏱️ Distribution of Time to Fix (Days)', fontsize=14, fontweight='bold')
plt.xlabel('Days')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../data/eda_time_to_fix.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚠️ Step 5: Identify High-Risk Modules
A module is high-risk if it has many bugs, many high-severity bugs, or many reopened bugs.

In [ ]:
# ── Risk Score Formula ────────────────────────────────────────────
# Risk = (High bugs × 3) + (Reopened × 2) + (Total occurrences)

severity_map = {'Low': 1, 'Medium': 2, 'High': 3}
status_map   = {'Open': 1, 'Fixed': 0, 'Reopened': 2}

df['Severity_Score'] = df['Severity'].map(severity_map)
df['Status_Score']   = df['Status'].map(status_map)

# Calculate per-module risk
module_risk = df.groupby('Module').agg(
    Total_Bugs        = ('Bug_ID', 'count'),
    High_Severity     = ('Severity', lambda x: (x == 'High').sum()),
    Reopened_Bugs     = ('Status',   lambda x: (x == 'Reopened').sum()),
    Avg_Occurrences   = ('Occurrences', 'mean'),
    Avg_Time_to_Fix   = ('Time_to_Fix_Days', 'mean')
).reset_index()

# Weighted risk score
module_risk['Risk_Score'] = (
    module_risk['High_Severity']   * 3 +
    module_risk['Reopened_Bugs']   * 2 +
    module_risk['Avg_Occurrences'] * 1
).round(2)

# Normalize 0–100
min_r = module_risk['Risk_Score'].min()
max_r = module_risk['Risk_Score'].max()
module_risk['Risk_Score_Normalized'] = (
    (module_risk['Risk_Score'] - min_r) / (max_r - min_r) * 100
).round(2)

module_risk = module_risk.sort_values('Risk_Score_Normalized', ascending=False)
print('🔥 High-Risk Modules:')
module_risk

In [ ]:
# ── Visualize Risk Scores ─────────────────────────────────────────
plt.figure(figsize=(12, 5))
colors = ['#e74c3c' if s >= 70 else '#f39c12' if s >= 40 else '#2ecc71'
          for s in module_risk['Risk_Score_Normalized']]
bars = plt.bar(module_risk['Module'], module_risk['Risk_Score_Normalized'],
               color=colors, edgecolor='black')

# Add value labels
for bar, val in zip(bars, module_risk['Risk_Score_Normalized']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}', ha='center', fontsize=9)

plt.axhline(70, color='red',    linestyle='--', label='High Risk (≥70)')
plt.axhline(40, color='orange', linestyle='--', label='Medium Risk (≥40)')
plt.title('⚠️ Module Risk Score (0–100)', fontsize=14, fontweight='bold')
plt.xlabel('Module')
plt.ylabel('Risk Score')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/module_risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 Step 6: Bug Risk Prediction Model
We train a Random Forest model to predict if a bug will be **Reopened** (high risk) or not.

In [ ]:
# ── 6a. Encode categorical features ──────────────────────────────
le_module   = LabelEncoder()
le_severity = LabelEncoder()
le_version  = LabelEncoder()

df['Module_Enc']   = le_module.fit_transform(df['Module'])
df['Severity_Enc'] = le_severity.fit_transform(df['Severity'])
df['Version_Enc']  = le_version.fit_transform(df['App_Version'])

# ── 6b. Target: 1 = Reopened (risky), 0 = Fixed/Open ─────────────
df['Target'] = (df['Status'] == 'Reopened').astype(int)

# ── 6c. Feature matrix ───────────────────────────────────────────
FEATURES = ['Module_Enc', 'Severity_Enc', 'Version_Enc',
            'Occurrences', 'Time_to_Fix_Days']
X = df[FEATURES]
y = df['Target']

# ── 6d. Split data ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

In [ ]:
# ── 6e. Train Random Forest ───────────────────────────────────────
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('📊 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Reopened', 'Reopened']))

In [ ]:
# ── 6f. Confusion Matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Reopened', 'Reopened'],
            yticklabels=['Not Reopened', 'Reopened'])
plt.title('🎯 Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6g. Feature Importance ────────────────────────────────────────
feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
feat_imp.plot(kind='bar', color='coral', edgecolor='black')
plt.title('🔍 Feature Importance', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 📝 Step 7: NLP — Detect Recurring Bug Descriptions
Using TF-IDF + Cosine Similarity to find bugs that describe similar issues.

In [ ]:
# ── Vectorize bug descriptions ────────────────────────────────────
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Bug_Description'])

# Compute similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

# Find top recurring bug pairs (similarity > 0.8, not same bug)
recurring_pairs = []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        sim = similarity_matrix[i][j]
        if sim > 0.8:
            recurring_pairs.append({
                'Bug_1': df.iloc[i]['Bug_ID'],
                'Bug_2': df.iloc[j]['Bug_ID'],
                'Description_1': df.iloc[i]['Bug_Description'],
                'Description_2': df.iloc[j]['Bug_Description'],
                'Similarity': round(sim, 3)
            })

recurring_df = pd.DataFrame(recurring_pairs).sort_values('Similarity', ascending=False)
print(f'🔁 Recurring bug pairs found: {len(recurring_df)}')
recurring_df.head(10)

## ✅ Step 8: Bug Fix Validation Logic
Logic to check if a previously fixed bug is likely to reappear.

In [ ]:
def validate_fix(row):
    """
    Returns 'Likely Resolved' or 'At Risk of Reappearing'
    based on severity, occurrences, and time to fix.
    """
    score = 0
    if row['Severity'] == 'High':               score += 3
    elif row['Severity'] == 'Medium':            score += 2
    else:                                        score += 1

    if row['Occurrences'] > 20:                  score += 3
    elif row['Occurrences'] > 10:                score += 2
    else:                                        score += 1

    if row['Time_to_Fix_Days'] > 15:             score += 2
    elif row['Time_to_Fix_Days'] > 7:            score += 1

    return 'At Risk of Reappearing' if score >= 6 else 'Likely Resolved'

# Apply only on Fixed bugs
fixed_bugs = df[df['Status'] == 'Fixed'].copy()
fixed_bugs['Validation'] = fixed_bugs.apply(validate_fix, axis=1)

print('Fix Validation Results:')
print(fixed_bugs['Validation'].value_counts())
fixed_bugs[['Bug_ID', 'Module', 'Severity', 'Occurrences',
             'Time_to_Fix_Days', 'Validation']].head(15)

## 📋 Step 9: Testing Priority for Next Release

In [ ]:
# Merge risk scores back to main df
df_priority = df.merge(module_risk[['Module', 'Risk_Score_Normalized']], on='Module')

# Priority score: risk + severity + occurrences
df_priority['Priority_Score'] = (
    df_priority['Risk_Score_Normalized'] * 0.5 +
    df_priority['Severity_Score']        * 15  +
    df_priority['Occurrences']           * 0.5
).round(2)

priority_list = (
    df_priority[df_priority['Status'] != 'Fixed']
    .sort_values('Priority_Score', ascending=False)
    [['Bug_ID', 'Module', 'Severity', 'Status',
      'Occurrences', 'Risk_Score_Normalized', 'Priority_Score']]
    .head(20)
)

print('🚨 Top 20 Bugs to Test in Next Release:')
priority_list

## 💡 Step 10: Actionable Insights & Recommendations

In [ ]:
top_module      = module_risk.iloc[0]['Module']
top_risk_score  = module_risk.iloc[0]['Risk_Score_Normalized']
reopen_rate     = round((df['Status'] == 'Reopened').mean() * 100, 1)
high_sev_pct    = round((df['Severity'] == 'High').mean() * 100, 1)
avg_ttf         = round(df['Time_to_Fix_Days'].mean(), 1)
latest_version  = sorted(df['App_Version'].unique())[-1]
latest_bugs     = df[df['App_Version'] == latest_version].shape[0]
prev_version    = sorted(df['App_Version'].unique())[-2]
prev_bugs       = df[df['App_Version'] == prev_version].shape[0]
trend           = 'improving ✅' if latest_bugs < prev_bugs else 'worsening ⚠️'

insights = f"""
╔══════════════════════════════════════════════════════════════╗
║          💡 10 ACTIONABLE INSIGHTS & RECOMMENDATIONS        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║ 1. ⚠️  '{top_module}' is the highest-risk module            ║
║    → Risk Score: {top_risk_score:.1f}/100. Prioritize testing here.    ║
║                                                              ║
║ 2. 🔁  Reopen rate is {reopen_rate}% — bugs are reoccurring ║
║    → Improve root cause analysis before closing bugs.       ║
║                                                              ║
║ 3. 🔴  {high_sev_pct}% of bugs are High severity           ║
║    → Allocate senior testers to high-severity modules.      ║
║                                                              ║
║ 4. ⏱️  Average fix time = {avg_ttf} days                    ║
║    → Set SLA: High bugs fixed in <5 days.                   ║
║                                                              ║
║ 5. 📈  Latest version ({latest_version}) is {trend}         ║
║    → {latest_bugs} bugs vs {prev_bugs} in {prev_version}.              ║
║                                                              ║
║ 6. 🔬  NLP shows many duplicate bug descriptions            ║
║    → Merge duplicate bugs to avoid redundant retesting.     ║
║                                                              ║
║ 7. 📊  Payment & Auth modules have high reopen rates        ║
║    → Add regression tests specifically for these modules.   ║
║                                                              ║
║ 8. 🎯  Use Priority Score to guide sprint test planning     ║
║    → Test top 20 priority bugs every release cycle.         ║
║                                                              ║
║ 9. 🧹  Use validation logic to avoid premature closure      ║
║    → Bugs with score ≥ 6 need extended monitoring.          ║
║                                                              ║
║ 10. 🤖 ML model can be retrained each release               ║
║     → Feed new bugs to improve prediction accuracy.         ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(insights)